# Sensitivity of the headline results to the segment definition

A full-throttle segment is throttle at or above `FULL_THROTTLE` (98) with the brake off, at least `MIN_SEGMENT_M` (500 m) long. Both numbers are judgement calls, so this reruns the segment extraction and lap selection (same code, `flatout.segment_grid`) with throttle thresholds of 95, 98 and 100 and minimum lengths of 300, 500 and 800 m, on every 2026 session and every paired 2025 session. The reporting rule stays the same (at least 10 clean drivers per straight)

For each combination you get the season headline numbers from `02_season.ipynb` and the paired 2025 vs 2026 headline from `03_regulations.ipynb`, so run those two first. Per-session results are cached in `data/sensitivity/`

In [1]:
import json
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

import fastf1
import numpy as np
import pandas as pd

from flatout import FULL_THROTTLE, MIN_SEGMENT_M, OPENF1_CACHE, caption, openf1_session_key, segment_grid

MIN_CLEAN_DRIVERS = 10
GRID = [(thr, m) for thr in (95, 98, 100) for m in (300, 500, 800)]
OUT = Path("data/sensitivity")
OUT.mkdir(parents=True, exist_ok=True)


def circuit_of(year, rnd):
    """(OpenF1 session_key, circuit_short_name) of an event's main Qualifying (as in 03_regulations.ipynb)."""
    key = openf1_session_key(year, rnd)
    ses = pd.DataFrame(json.loads((OPENF1_CACHE / f"sessions_{year}_qualifying.json").read_text()))
    row = ses[ses.session_key == key]
    return key, (int(row.circuit_key.iloc[0]) if len(row) else None), (row.circuit_short_name.iloc[0] if len(row) else None)


qc26 = pd.read_csv("data/season/qc_summary.csv")
qc26 = qc26[qc26.Status.eq("ok")]
s26 = pd.DataFrame([dict(Year=2026, Round=r.Round, Event=r.Event, **dict(zip(["Key", "CircuitKey", "CircuitName"], circuit_of(2026, r.Round))))
                    for r in qc26.itertuples()])
sch25 = fastf1.get_event_schedule(2025, include_testing=False)
sch25 = sch25[sch25.EventFormat.isin(["conventional", "sprint_qualifying"])]
s25 = pd.DataFrame([dict(Year=2025, Round=r.RoundNumber, Event=r.EventName, **dict(zip(["Key", "CircuitKey", "CircuitName"], circuit_of(2025, r.RoundNumber))))
                    for r in sch25.itertuples()])
paired_keys = set(s26[~s26.Event.eq("Bahrain Grand Prix")].CircuitKey)  # 2026 Bahrain GP was held at Sepang (as in 03)
s25 = s25[s25.CircuitKey.isin(paired_keys)]
sessions = pd.concat([s26, s25], ignore_index=True)
print(f"{len(s26)} sessions in 2026, {len(s25)} paired sessions in 2025; grid: {GRID}")

req         WARNING 	DEFAULT CACHE ENABLED! (56.0 KB) C:\Users\User\AppData\Local\Temp\fastf1


17 sessions in 2026, 15 paired sessions in 2025; grid: [(95, 300), (95, 500), (95, 800), (98, 300), (98, 500), (98, 800), (100, 300), (100, 500), (100, 800)]


In [2]:
paths = [OUT / f"{r.Year}_{r.Round:02d}_segments.csv" for r in sessions.itertuples()]
todo = [(p, r) for p, r in zip(paths, sessions.itertuples()) if not p.exists()]  # cached: delete data/sensitivity/ to recompute
with ProcessPoolExecutor(max_workers=4) as pool:  # segment_grid is importable, so worker processes can run it
    for (p, _), seg_ in zip(todo, pool.map(segment_grid, *zip(*[(r.Year, r.Round, r.Key, GRID) for _, r in todo]))):
        seg_.to_csv(p, index=False)
grid_seg = pd.concat([pd.read_csv(p).assign(Year=r.Year, Round=r.Round, Event=r.Event, CircuitName=r.CircuitName, CircuitKey=r.CircuitKey)
                      for p, r in zip(paths, sessions.itertuples())], ignore_index=True)
print(f"{len(grid_seg)} selected segments across {len(GRID)} combinations")

15074 selected segments across 9 combinations


In [3]:
def headline_numbers(d):
    """Season (2026) and paired (2025 vs 2026) headline numbers for one combination's selected segments."""
    s = d[d.Year.eq(2026)].copy()
    s["Straight"] = s.Round.map("{:02d}".format) + " " + s.Event.str.replace(" Grand Prix", "") + " " + np.where(s.CrossesTimingLine, "main", s.Label)
    rep = s[s.groupby("Straight").Driver.transform("nunique") >= MIN_CLEAN_DRIVERS]
    p = d[d.CircuitKey.isin(paired_keys) & ~(d.Year.eq(2026) & d.Event.eq("Bahrain Grand Prix"))].copy()
    p["Straight"] = p.CircuitKey.astype(str) + " " + np.where(p.CrossesTimingLine, "main", p.Label)  # matched by circuit_key, as in 03
    p = p[p.groupby(["Straight", "Year"]).Driver.transform("nunique") >= MIN_CLEAN_DRIVERS]
    per = p.groupby(["Straight", "Year"])[["vmax_position", "clip_delta"]].median().unstack("Year").dropna()
    diff = pd.DataFrame({m: per[(m, 2026)] - per[(m, 2025)] for m in ["vmax_position", "clip_delta"]})
    no_clip = (per[("clip_delta", 2025)] == 0) & (per[("clip_delta", 2026)] == 0)
    dc = diff[~no_clip]
    names = p.drop_duplicates("Straight").set_index("Straight").CircuitName  # readable name for the circuit_key-based key
    rev = dc[(dc.vmax_position > 0) | (dc.clip_delta < 0)]
    return {"Reverse straights": "; ".join(f"{names[k]} {k.split(' ', 1)[1]} (2025 vs 2026: vmax_position {per.loc[k, ('vmax_position', 2025)]:.2f} vs "
                                           f"{per.loc[k, ('vmax_position', 2026)]:.2f}, clip_delta {per.loc[k, ('clip_delta', 2025)]:.1f} vs "
                                           f"{per.loc[k, ('clip_delta', 2026)]:.1f} km/h)" for k in rev.index),
"Driver-straights 2026": len(rep), "Reportable straights 2026": rep.Straight.nunique(),
            "Peak before end (%)": 100 * (rep.vmax_position < 1).mean(), "Median vmax_position": rep.vmax_position.median(),
            "Matched straights": len(per), "With clipping": len(dc),
            "Earlier and harder": int(((dc.vmax_position < 0) & (dc.clip_delta > 0)).sum()),
            "Reverse": int(((dc.vmax_position > 0) | (dc.clip_delta < 0)).sum()),
            "Median clip_delta difference (km/h)": diff.clip_delta.median()}


sens = pd.DataFrame([dict(Throttle=thr, MinLength_m=m, **headline_numbers(g)) for (thr, m), g in grid_seg.groupby(["Throttle", "MinLength_m"])])
base = sens[(sens.Throttle == FULL_THROTTLE) & (sens.MinLength_m == MIN_SEGMENT_M)].iloc[0]
assert (base["Driver-straights 2026"], base["Reportable straights 2026"], base["Matched straights"], base["With clipping"], base["Earlier and harder"]) \
    == (842, 41, 39, 33, 33), "baseline does not reproduce 02_season / 03_regulations"
sens.drop(columns="Reverse straights").round(2)

,Throttle,MinLength_m,Driver-straights 2026,Reportable straights 2026,Peak before end (%),Median vmax_position,Matched straights,With clipping,Earlier and harder,Reverse,Median clip_delta difference (km/h)
0,95,300,1320,63,70.15,0.88,61,46,45,1,4.00
1,95,500,845,41,85.68,0.76,40,34,34,0,8.00
2,95,800,395,19,94.94,0.66,18,18,18,0,19.75
3,98,300,1316,63,70.14,0.88,60,44,44,0,4.00
4,98,500,842,41,85.63,0.76,39,33,33,0,7.50
5,98,800,395,19,94.68,0.66,18,18,18,0,19.50
6,100,300,1245,63,68.59,0.88,59,44,44,0,3.00
7,100,500,795,41,84.78,0.77,39,33,33,0,7.00
8,100,800,370,19,94.32,0.66,18,18,18,0,20.00


*Table 1. Headline numbers under 9 segment definitions (throttle threshold 95, 98, 100 %, minimum length 300, 500, 800 m; baseline 98 %, 500 m). Share of 2026 driver-straights peaking before the segment end: 69–95 %; median vmax_position 0.66–0.88. All straights with clipping peak earlier and lose more speed in 2026 in 8 of 9 combinations; at 95 %, 300 m: 45 of 46, exception Silverstone T16 (2025 vs 2026: vmax_position 0.97 vs 1.00, clip_delta 0.5 vs 0.0 km/h) (18–46 straights with clipping of 18–61 matched); median clip_delta difference +3.0 to +20.0 km/h. Descriptive*